In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, get_json_object, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType, TimestampType
from pyspark.sql import functions as F

In [2]:
try:
    spark.stop()
except:
    pass

In [3]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_ExplanationOfBenefits")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 11:22:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=ExplanationOfBenefit"
silver_eob_header_path = "../../data_lake/silver/silver_eob_header/"
silver_eob_line_item_path = "../../data_lake/silver/silver_eob_line_item/"

In [5]:
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])


telecom_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True),
        StructField("use", StringType(), True)
])

address_schema = StructType([
        StructField("line", ArrayType(StringType()), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True),
        StructField("country", StringType(), True)
    ])

identifier_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True)
    ])

extension_schema = StructType([
        StructField("url", StringType(), True),
        StructField("valueInteger", StringType(), True)
])

name_schema = StructType([
        StructField("family", StringType(), True),
        StructField("given", ArrayType(StringType()), True),
        StructField("prefix", ArrayType(StringType()), True)
])

billablePeriod_schema = StructType([
        StructField("start", TimestampType(), True),
        StructField("end", TimestampType(), True)
])

careTeam_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("provider", StringType(), True),
        StructField("role", type_schema, True),
        # StructField("qualification", type_schema, True)
])

diagnosis_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("diagnosisReference", StringType(), True),
        StructField("type", type_schema, True)
])

insurance_schema = StructType([
        StructField("focal", StringType(), True),
        StructField("coverage", StructType([
            StructField("reference", StringType(), True),
            StructField("display", StringType(), True)
        ]), True)
])

amount_schema = StructType([
        StructField("value", StringType(), True),
        StructField("currency", StringType(), True)
])

adjudication_schema = StructType([
        StructField("category", type_schema, True),
        StructField("amount", amount_schema, True)
])

item_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("diagnosisSequence", ArrayType(StringType()), True),
        StructField("informationSequence", ArrayType(StringType()), True),
        StructField("category", type_schema, True),
        StructField("productOrService", type_schema, True),
        StructField("servicedPeriod", billablePeriod_schema, True),
        StructField("locationCodeableConcept", type_schema, True),
        StructField("net", amount_schema, True),
        StructField("encounter", ArrayType(MapType(StringType(), StringType())), True),
        StructField("adjudication", ArrayType(adjudication_schema), True)
])

payment_schema = StructType([
        StructField("amount", amount_schema, True)
])

total_schema = StructType([
        StructField("category", type_schema, True),
        StructField("amount", amount_schema, True)
])

In [6]:
df_eob = (spark.read.format("parquet").load(bronze_path))

In [7]:
df_inter1 = (
    df_eob.select(
        col("resource.id").alias("eob_id"),
        # col("resource.contained").alias("contained"),
        from_json(col("resource.identifier"), ArrayType(identifier_schema)).alias("identifier"),
        col("resource.status").alias("status"),
        from_json(col("resource.type"), type_schema).alias("type"),
        col("resource.use").alias("use"),
        F.split(get_json_object(col("resource.patient"), "$.reference"), ":")[2].alias("patient_id"),
        from_json(col("resource.billablePeriod"), billablePeriod_schema).alias("billablePeriod"),
        col("resource.created").cast("timestamp").alias("created"),
        get_json_object(col("resource.insurer"), "$.display").alias("insurer"),
        F.split(get_json_object(col("resource.provider"), "$.reference"), "\|")[1].alias("provider"),
        get_json_object(col("resource.referral"), "$.reference").alias("referral"),
        F.split(get_json_object(col("resource.facility"), "$.reference"), "\|")[1].alias("facility"),
        F.split(get_json_object(col("resource.claim"), "$.reference"), ":")[2].alias("claim"),
        col("resource.outcome").alias("outcome"),
        from_json(col("resource.careTeam"), ArrayType(careTeam_schema)).alias("careTeam"),
        from_json(col("resource.diagnosis"), ArrayType(diagnosis_schema)).alias("diagnosis"),
        # col("resource.procedure").alias("procedure"),
        from_json(col("resource.insurance"), ArrayType(insurance_schema)).alias("insurance"),
        from_json(col("resource.item"), ArrayType(item_schema)).alias("item"),
        from_json(col("resource.total"), ArrayType(total_schema)).alias("total"),
        from_json(col("resource.payment"), payment_schema).alias("payment")    
)
)

In [8]:
silver_eob_header = df_inter1.select(
    col("eob_id"),
    col("status"),
    col("use"),
    col("outcome"),
    col("claim").alias("claim_id"),        

    # ── Identifier ────────────────────────────────────────
    # CMS claim ID = identifier[0], claim group = identifier[1]
    col("identifier")[0]["value"].alias("cms_claim_id"),
    col("identifier")[1]["value"].alias("cms_claim_group_id"),

    # ── Claim type ────────────────────────────────────────
    col("type.coding")[0]["code"].alias("claim_type"),  # professional | institutional

    # ── Parties ───────────────────────────────────────────
    col("patient_id"),                        # already clean UUID
    col("provider").alias("practitioner_npi"),# already extracted NPI
    col("insurer").alias("payer_name"),        # Medicare | Medicaid | NO_INSURANCE
    col("facility").alias("facility_synthea_id"),
    col("referral"),

    # ── Dates ─────────────────────────────────────────────
    col("billablePeriod.start").cast("timestamp").alias("service_start_dt"),
    col("billablePeriod.end").cast("timestamp").alias("service_end_dt"),
    col("created").alias("created_dt"),

    # ── Financial — header level ──────────────────────────
    # total is an array — find the "submitted" entry
    F.filter(col("total"), lambda x:
        x["category"]["coding"][0]["code"] == "submitted"
    )[0]["amount"]["value"].alias("total_submitted_amt"),

    col("payment.amount.value").alias("payment_amt"),

    # ── Insurance ─────────────────────────────────────────
    col("insurance")[0]["focal"].alias("is_focal_insurance"),
    col("insurance")[0]["coverage"]["reference"].alias("coverage_reference"),
    col("insurance")[0]["coverage"]["display"].alias("coverage_display"),

    # ── Derived counts ────────────────────────────────────
    F.size(col("diagnosis")).alias("diagnosis_count"),
    F.size(col("careTeam")).alias("care_team_count"),
    F.size(col("item")).alias("line_item_count"),
)

In [9]:
df_items = df_inter1.select(
    col("eob_id"),
    col("claim").alias("claim_id"),
    col("patient_id"),
    explode_outer(col("item")).alias("item")
)

silver_eob_line_item = df_items.select(

    # ── Keys ──────────────────────────────────────────────
    col("eob_id"),
    col("claim_id"),
    col("patient_id"),
    col("item.sequence").alias("item_seq"),

    # ── Item type (derived) ───────────────────────────────
    F.when(col("item.encounter").isNotNull(),        "encounter")
     .when(col("item.diagnosisSequence").isNotNull(), "diagnosis")
     .when(col("item.informationSequence").isNotNull(),"observation")
     .otherwise("procedure")
     .alias("item_type"),

    # ── Sequence references ───────────────────────────────
    col("item.diagnosisSequence")[0].alias("diagnosis_seq_ref"),
    col("item.informationSequence")[0].alias("info_seq_ref"),

    # ── Service code ──────────────────────────────────────
    col("item.productOrService.coding")[0]["code"].alias("service_code"),
    col("item.productOrService.coding")[0]["system"].alias("service_code_system"),
    col("item.productOrService.text").alias("service_description"),

    # ── CMS service type ──────────────────────────────────
    col("item.category.coding")[0]["code"].alias("cms_service_type_code"),
    col("item.category.coding")[0]["display"].alias("cms_service_type_display"),

    # ── Service location ──────────────────────────────────
    col("item.locationCodeableConcept.coding")[0]["code"].alias("service_place_code"),
    col("item.locationCodeableConcept.coding")[0]["display"].alias("service_place_display"),

    # ── Dates ─────────────────────────────────────────────
    col("item.servicedPeriod.start").cast("timestamp").alias("service_start_dt"),
    col("item.servicedPeriod.end").cast("timestamp").alias("service_end_dt"),

    # ── Financial — net ───────────────────────────────────
    col("item.net.value").alias("net_amt"),
    col("item.net.currency").alias("currency"),

    # ── Financial — adjudication (pivot array → columns) ──
    F.filter(col("item.adjudication"), lambda x:
        x["category"]["coding"][0]["code"].endswith("line_sbmtd_chrg_amt")
    )[0]["amount"]["value"].alias("submitted_amt"),

    F.filter(col("item.adjudication"), lambda x:
        x["category"]["coding"][0]["code"].endswith("line_alowd_chrg_amt")
    )[0]["amount"]["value"].alias("allowed_amt"),

    F.filter(col("item.adjudication"), lambda x:
        x["category"]["coding"][0]["code"].endswith("line_prvdr_pmt_amt")
    )[0]["amount"]["value"].alias("provider_payment_amt"),

    F.filter(col("item.adjudication"), lambda x:
        x["category"]["coding"][0]["code"].endswith("line_coinsrnc_amt")
    )[0]["amount"]["value"].alias("coinsurance_amt"),

    F.filter(col("item.adjudication"), lambda x:
        x["category"]["coding"][0]["code"].endswith("line_bene_ptb_ddctbl_amt")
    )[0]["amount"]["value"].alias("deductible_amt"),

    # ── Flags ─────────────────────────────────────────────
    col("item.net").isNotNull().alias("is_billable"),
)

In [10]:
# pdf = silver_eob_line_item.limit(10).toPandas()

# display_pdf = pdf.style.set_properties(**{'text-align': 'left'}).set_table_styles([{
#     'selector': 'th',
#     'props': [('text-align', 'left')]
# }])

# display(display_pdf)

In [11]:
silver_eob_header.write.mode("overwrite").format("parquet").save(silver_eob_header_path)
silver_eob_line_item.write.mode("overwrite").format("parquet").save(silver_eob_line_item_path)

In [12]:
spark.stop()